# Dripper / MinerU-HTML Layout Clustering Tutorial

This notebook walks through the complete pipeline step-by-step, using a real slice of CC-MAIN-2025-26.

**The core idea**: running LLM extraction on every Common Crawl HTML page is expensive (~242K H100-hours for one snapshot). Most pages on the same website share the same DOM layout. We can:
1. Cluster pages by DOM structure (CPU, cheap)
2. Run LLM on one representative per cluster (GPU, expensive)
3. Apply the LLM's decisions as a template to all siblings (CPU, cheap)

**Data**: 8192 pages from 16 hosts in CC-MAIN-2025-26, pre-clustered.  
**Model**: `opendatalab/MinerU-HTML-v1.1-hunyuan0.5B-compact` (0.5B, fits on 1× A100).

---
## Sections
0. Setup
1. Load data — look at raw HTML pages  
2. DOM feature extraction — how we fingerprint page structure  
3. Layout clustering — DBSCAN groups similar-structure pages  
4. Representative selection — which page in a cluster to run LLM on  
5. HTML simplification — what the LLM actually sees  
6. LLM extraction — MinerU-HTML labels nodes main/non-main  
7. Template propagation — apply labels to siblings without GPU  
8. Validation — measure F1 vs pure Dripper baseline  
9. Cost analysis — how much GPU time we save  
10. Full pipeline — `DripperHTMLExtractionPipelineStage` end-to-end  

## 0. Setup

In [ ]:
import sys

# Paths on dgx-a100-02
CURATOR_REPO = "/raid/vjawa/nemo-curator-adlr-mm/submodules/Curator"
DATA_DIR     = "/raid/vjawa/dripper_tutorial"

print(f"Data dir:     {DATA_DIR}")
print(f"Curator repo: {CURATOR_REPO}")

In [ ]:
import os, sys
sys.path.insert(0, CURATOR_REPO)

import pandas as pd
import numpy as np
import json
import re
import pyarrow.parquet as pq
import IPython.display as display
from collections import Counter
from pathlib import Path

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

def read_parquet_safe(path):
    """
    Read a parquet file using pyarrow.parquet.ParquetFile directly.
    Avoids the ParquetDataset memory-map buffer issue that causes:
      ArrowInvalid: Parquet magic bytes not found in footer
    """
    return pq.ParquetFile(str(path)).read().to_pandas()

print("Imports OK — read_parquet_safe() available")

## 1. Load Data — Raw HTML Pages

The input is a parquet with one row per CC page. Key columns:
- `url` — page URL
- `url_host_name` — hostname (used for locality)
- `html` — raw HTML bytes
- `dripper_layout_id` — pre-assigned layout cluster ID (from a prior CPU clustering pass)

In [ ]:
manifest = read_parquet_safe(f"{DATA_DIR}/layout_precompute_manifest.parquet")
print(f"Manifest: {len(manifest):,} pages, {manifest['url_host_name'].nunique()} unique hosts")

# Baseline is optional — sections 6–8 need it, rest works without it
try:
    baseline = read_parquet_safe(f"{DATA_DIR}/baseline_dripper_results.parquet")
    print(f"Baseline: {len(baseline):,} rows — F1 comparison cells available")
except Exception as e:
    baseline = None
    print(f"⚠ Baseline not loaded ({e.__class__.__name__}: {e!s:.80})")
    print("  Re-run: rsync -az vjawa@nb-hel-cs-001-dc-01.nvidia.com:/lustre/fsw/portfolios/llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/328281/dripper_results.parquet /raid/vjawa/dripper_tutorial/baseline_dripper_results.parquet")

print()
host_counts = manifest['url_host_name'].value_counts()
print("Pages per host:")
print(host_counts.to_string())

In [ ]:
# Look at a few raw HTML pages
sample = manifest.sample(3, random_state=42)
for _, row in sample.iterrows():
    html_bytes = row['html']
    if isinstance(html_bytes, bytes):
        html_str = html_bytes.decode('utf-8', errors='replace')
    else:
        html_str = str(html_bytes)
    print(f"URL: {row['url']}")
    print(f"Host: {row['url_host_name']}")
    print(f"Layout ID: {row['dripper_layout_id']}")
    print(f"HTML size: {len(html_str):,} chars")
    print(f"HTML preview: {html_str[:200].strip()!r}")
    print()

In [ ]:
# Render one page in the notebook
row = manifest[manifest['url_host_name'] == 'scratch.mit.edu'].iloc[0]
html_str = row['html'].decode('utf-8', errors='replace') if isinstance(row['html'], bytes) else str(row['html'])
print(f"Rendering: {row['url']}")
display.display(display.HTML(f'<iframe srcdoc="{html_str[:5000].replace(chr(34), chr(39))}" width="900" height="400" style="border:1px solid #ccc"></iframe>'))

## 2. DOM Feature Extraction

The `get_feature()` function from `llm-webkit` extracts a structural fingerprint of a page:
- Traverses the DOM tree layer by layer
- Records tag names + class/id attributes per depth
- Ignores noisy tags (`script`, `style`, `meta`, `link`)
- Normalizes dynamic attributes (removes hashes, UUIDs, timestamps)

This gives a compact representation of page structure independent of content.

In [ ]:
# Load llm-webkit bindings via Curator's helper
from nemo_curator.stages.text.experimental.dripper.stage import _load_llm_web_kit_bindings
web = _load_llm_web_kit_bindings()
print("llm-webkit bindings loaded")
print(f"  cluster_html_struct: {web.cluster_html_struct}")
print(f"  get_feature: {web.get_feature}")
print(f"  select_representative_html: {web.select_representative_html}")

In [ ]:
def coerce_html(raw):
    if isinstance(raw, bytes):
        return raw.decode('utf-8', errors='replace')
    return str(raw or '')

# Extract features from 3 pages on the same host — should look similar
host_rows = manifest[manifest['url_host_name'] == 'hysplitbbs.arl.noaa.gov'].head(3)

print("Features from 3 pages on hysplitbbs.arl.noaa.gov:")
print("(Same host = very similar DOM structure)")
print()
for _, row in host_rows.iterrows():
    html = coerce_html(row['html'])
    feat = web.get_feature(html)
    if feat:
        n_layers = len(feat.get('tags', {}))
        total_tags = sum(len(v) for v in feat.get('tags', {}).values())
        print(f"URL: ...{row['url'][-60:]}")
        print(f"  Layers: {n_layers}, Total tag entries: {total_tags}")
        # Show first 2 layers
        for layer_idx in sorted(feat.get('tags', {}).keys())[:2]:
            tags = feat['tags'][layer_idx][:5]
            print(f"  Layer {layer_idx}: {tags}")
        print()

In [ ]:
# Now compare with pages from a different host — features should differ
print("Features from gen.medium.com (different structure):")
medium_rows = manifest[manifest['url_host_name'] == 'gen.medium.com'].head(2)
for _, row in medium_rows.iterrows():
    html = coerce_html(row['html'])
    feat = web.get_feature(html)
    if feat:
        n_layers = len(feat.get('tags', {}))
        total_tags = sum(len(v) for v in feat.get('tags', {}).values())
        print(f"URL: ...{row['url'][-60:]}")
        print(f"  Layers: {n_layers}, Total tag entries: {total_tags}")
        for layer_idx in sorted(feat.get('tags', {}).keys())[:2]:
            tags = feat['tags'][layer_idx][:5]
            print(f"  Layer {layer_idx}: {tags}")
        print()

## 3. Layout Clustering

`cluster_html_struct()` runs DBSCAN over the DOM features:
- Computes pairwise cosine similarity (tag weight=0.7, attr weight=0.3)
- DBSCAN with eps=1-threshold (default threshold=0.95)
- Pages within the same host get `layout_id` 0,1,2... or -1 (noise)

The key constraint: clustering runs **within each host** — cross-host mixing never happens.

In [ ]:
# Cluster one host from scratch to see DBSCAN in action
host = 'scratch.mit.edu'
host_rows = manifest[manifest['url_host_name'] == host].head(50)

samples = []
for i, (_, row) in enumerate(host_rows.iterrows()):
    html = coerce_html(row['html'])
    feat = web.get_feature(html)
    if feat:
        samples.append({'track_id': str(i), 'html': html, 'feature': feat})

print(f"Extracted features for {len(samples)} pages")
clustered, layout_ids = web.cluster_html_struct(samples, threshold=0.95)

# Show cluster assignment distribution
id_counts = Counter(s['layout_id'] for s in clustered)
print(f"\nLayout cluster distribution (50 pages from {host}):")
for lid, count in sorted(id_counts.items(), key=lambda x: -x[1]):
    label = f"cluster-{lid}" if lid >= 0 else "noise (unique pages)"
    bar = '█' * count
    print(f"  {label:20s}: {count:3d} {bar}")

In [ ]:
# Show URLs in the largest cluster — they should look structurally identical
largest_cluster_id = max(id_counts, key=lambda x: id_counts[x] if x >= 0 else 0)
print(f"\nURLs in largest cluster (layout_id={largest_cluster_id}):")
for s in clustered:
    if s['layout_id'] == largest_cluster_id:
        orig_row = host_rows.iloc[int(s['track_id'])]
        print(f"  {orig_row['url']}")

print("\nThese pages share the same DOM structure → one LLM call covers all of them.")

In [ ]:
# Visualize the precomputed global clusters
import matplotlib.pyplot as plt

named = manifest[manifest['dripper_layout_id'].str.startswith('layout-', na=False)]
failed = manifest[~manifest['dripper_layout_id'].str.startswith('layout-', na=False)]
vc = named['dripper_layout_id'].value_counts()

bins = [2,5,10,25,50,100,250,600]
labels = [f'{bins[i]}-{bins[i+1]-1}' for i in range(len(bins)-1)]
counts = [((vc >= bins[i]) & (vc < bins[i+1])).sum() for i in range(len(bins)-1)]
pages  = [int(vc[(vc >= bins[i]) & (vc < bins[i+1])].sum()) for i in range(len(bins)-1)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.bar(labels, counts, color='steelblue')
ax1.set_title('Number of clusters by size')
ax1.set_xlabel('Cluster size (pages)')
ax1.set_ylabel('Clusters')
ax1.tick_params(axis='x', rotation=30)

ax2.bar(labels, pages, color='orange')
ax2.bar(['failed'], [len(failed)], color='red')
ax2.set_title('Pages by cluster size + failed')
ax2.set_xlabel('Cluster size')
ax2.set_ylabel('Pages')
ax2.tick_params(axis='x', rotation=30)

fig.suptitle(f'Global clustering: {len(named):,} clustered, {len(failed):,} failed (no layout)', y=1.02)
plt.tight_layout()
plt.show()
print(f"Total: {len(manifest):,} pages → {named['dripper_layout_id'].nunique()} clusters")
print(f"Potential savings ceiling: {len(named)/len(manifest)*100:.1f}% of pages are in clusters")

## 4. Representative Selection

For each layout cluster we pick the **best representative** — the page that most completely covers the layout's structural vocabulary. The scorer uses:
- XPath coverage (fraction of the cluster's unique XPaths this page contains)
- Tag count, tag diversity, max depth, avg width, width entropy

Formula: `score = 0.4 × coverage + 0.3 × structure_score + 0.3 × distribution_score`

In [ ]:
# Select a representative from the largest cluster
biggest_cluster_id = vc.index[0]
cluster_rows = manifest[manifest['dripper_layout_id'] == biggest_cluster_id].head(20)
print(f"Cluster: {biggest_cluster_id}")
print(f"Host: {cluster_rows['url_host_name'].iloc[0]}")
print(f"Size: {len(vc)} total, showing 20")

candidates = []
for _, row in cluster_rows.iterrows():
    html = coerce_html(row['html'])
    if html.strip():
        candidates.append({'track_id': row['url'], 'html': html})

rep = web.select_representative_html(candidates)
if rep:
    print(f"\nSelected representative URL: {rep.get('track_id')}")
    # Show why it was chosen vs a random candidate
    print("This page has the highest structural coverage score — best choice to run LLM on")
else:
    print("Fallback: using first candidate")

## 5. HTML Simplification — What the LLM Sees

Before sending to the LLM, Dripper **simplifies** the HTML:
- Removes non-content tags (`script`, `style`, `header`, `aside`)
- Keeps only `class` and `id` attributes  
- Truncates long text (paragraphs to first 200 chars)
- Assigns `_item_id` to each node for mapping labels back

Result: from ~50K tokens → ~7K tokens (12.83% of original). This makes the LLM fast and cheap.

In [ ]:
from nemo_curator.stages.text.experimental.dripper.stage import (
    _load_mineru_html_bindings,
    DripperHTMLExtractionStage,
)
import time

bindings = _load_mineru_html_bindings()
print("MinerU-HTML bindings loaded")

def simplify_html(bindings, raw_html, url=""):
    """Simplify raw HTML using MinerU-HTML — returns (simplified_html, mapped_html)."""
    case = bindings.case_cls(bindings.input_cls(raw_html=raw_html, url=url))
    case = bindings.simplify_single_input(case)
    simplified = DripperHTMLExtractionStage._get_processed_attr(case, "simpled_html")
    mapped     = DripperHTMLExtractionStage._get_processed_attr(case, "map_html")
    return simplified, mapped

# Demo: simplify a page and show the token reduction
sample_row = manifest[manifest['url_host_name'] == 'hysplitbbs.arl.noaa.gov'].iloc[0]
raw_html = coerce_html(sample_row['html'])

t0 = time.perf_counter()
simplified_html, mapped_html = simplify_html(bindings, raw_html, url=sample_row['url'])
elapsed = time.perf_counter() - t0

print(f"\nPage: {sample_row['url']}")
print(f"Raw HTML:        {len(raw_html):>8,} chars")
print(f"Simplified HTML: {len(simplified_html):>8,} chars  ({len(simplified_html)/max(len(raw_html),1)*100:.1f}% of original)")
print(f"Mapped HTML:     {len(mapped_html):>8,} chars")
print(f"Time:            {elapsed*1000:.0f}ms")
print()
print("Simplified HTML (first 600 chars):")
print(simplified_html[:600])

In [ ]:
print("Mapped HTML (first 600 chars) — each node gets an _item_id:")
print(mapped_html[:600])
item_ids = re.findall(r'_item_id="(\d+)"', mapped_html)
print(f"\nTotal nodes with _item_id: {len(item_ids)}")
print("These IDs are what the LLM labels as 'main' or 'other'")

## 6. LLM Extraction — MinerU-HTML Labels Nodes

The 0.5B model (`MinerU-HTML-v1.1-hunyuan0.5B-compact`) receives the simplified HTML and outputs a JSON dict:
```json
{"1": "main", "2": "other", "3": "main", ...}
```

- `"main"` = this node's content should be in the output
- `"other"` = nav, ads, boilerplate — skip

Constrained decoding enforces valid JSON — the model only picks between two tokens per item.

In [ ]:
if baseline is None:
    print("⚠  Baseline not loaded — run the rsync command from cell 1 to load it.")
else:
    baseline_merged = manifest.merge(
        baseline[['url','dripper_html','dripper_content','dripper_error','dripper_response']],
        on='url', how='left'
    )
    rep_url = rep['track_id'] if rep else cluster_rows['url'].iloc[0]
    rep_result = baseline_merged[baseline_merged['url'] == rep_url]

    if len(rep_result) and pd.notna(rep_result.iloc[0]['dripper_response']):
        raw_resp = rep_result.iloc[0]['dripper_response']
        print(f"LLM response for representative page:")
        print(f"URL: {rep_url}")
        print(f"Response: {str(raw_resp)[:400]}")
        print()
        content = rep_result.iloc[0]['dripper_content']
        print(f"Extracted content ({len(str(content))} chars):")
        print(str(content)[:600])
    else:
        print("Representative page not in baseline. Showing another example.")
        has_response = baseline_merged[baseline_merged['dripper_response'].notna()].head(1)
        if len(has_response):
            row = has_response.iloc[0]
            print(f"URL: {row['url']}")
            print(f"Response: {str(row['dripper_response'])[:400]}")
            print(f"\nContent: {str(row['dripper_content'])[:600]}")

In [ ]:
if baseline is None:
    print("⚠  Baseline not loaded — skipping token distribution stats.")
else:
    merged = manifest.merge(baseline[['url','dripper_prompt_tokens','dripper_completion_tokens',
                                       'dripper_time_s','dripper_error']], on='url', how='left')
    valid = merged[merged['dripper_error'].isna() | (merged['dripper_error'] == '')]
    print(f"Pages with successful extraction: {len(valid):,} / {len(merged):,}")
    print()
    print("Token usage distribution:")
    print(valid[['dripper_prompt_tokens','dripper_completion_tokens']].describe().round(0))
    print()
    print(f"Total tokens for 8192 pages: {valid['dripper_prompt_tokens'].sum() + valid['dripper_completion_tokens'].sum():,.0f}")
    print(f"Mean inference time: {valid['dripper_time_s'].mean():.2f}s per page")

## 7. Template Propagation — Apply to Siblings Without GPU

Once we have the representative's LLM labels, we distill them into a **structural template**:
- For each labeled node: record `(tag, class, id, depth, parent)` → `label`
- `LayoutBatchParser` walks a sibling page's DOM tree
- Matches nodes by structure (with fallbacks for dynamic IDs/classes)
- Extracts the same main content without any GPU call

This is the expensive CPU step (~11s/page) — the key bottleneck we're fixing with deferred propagation.

In [ ]:
# Find a cluster with multiple pages in baseline, pick representative and sibling
named_merged = baseline_merged[
    baseline_merged['dripper_layout_id'].str.startswith('layout-', na=False) &
    baseline_merged['dripper_content'].notna()
].copy()

cluster_sizes = named_merged.groupby('dripper_layout_id').size()
good_clusters = cluster_sizes[cluster_sizes >= 5].index
demo_cluster_id = good_clusters[0] if len(good_clusters) else named_merged['dripper_layout_id'].value_counts().index[0]

demo_cluster = named_merged[named_merged['dripper_layout_id'] == demo_cluster_id].copy()
print(f"Demo cluster: {demo_cluster_id}")
print(f"Host: {demo_cluster['url_host_name'].iloc[0]}")
print(f"Pages with baseline results: {len(demo_cluster)}")
print()
for _, row in demo_cluster.head(5).iterrows():
    print(f"  {row['url'][-80:]}")

In [ ]:
import time

# Build mapping_data from representative
rep_row = demo_cluster.iloc[0]
rep_html = coerce_html(rep_row['html'])

t0 = time.perf_counter()
simplified, mapped = simplify_html(bindings, rep_html, url=str(rep_row.get('url', '')))
simplify_time = time.perf_counter() - t0

# Simulate getting LLM response from baseline
rep_response = str(rep_row.get('dripper_response', '') or '')
if not rep_response:
    print("No LLM response for this rep; picking one that has it...")
    alt = demo_cluster[demo_cluster['dripper_response'].notna()]
    if len(alt):
        rep_row = alt.iloc[0]
        rep_html = coerce_html(rep_row['html'])
        simplified, mapped = simplify_html(bindings, rep_html, url=str(rep_row.get('url', '')))
        rep_response = str(rep_row['dripper_response'])

# Build the element_dict (template) via MapItemToHtmlTagsParser
t0 = time.perf_counter()
mapping_result = web.map_parser_cls({}).parse({
    'html_source': rep_html,
    'typical_raw_tag_html': mapped,
    'model_output': rep_response,
})
mapping_time = time.perf_counter() - t0

print(f"Simplification: {simplify_time*1000:.1f}ms")
print(f"Mapping (item→node): {mapping_time*1000:.1f}ms")
print(f"Mapping success: {mapping_result.get('typical_main_html_success')}")
print(f"Template HTML size: {len(str(mapping_result.get('typical_main_html',''))):,} chars")

In [ ]:
# Now propagate to a sibling page — NO GPU needed
sibling_row = demo_cluster.iloc[1]  # second page in same cluster
sibling_html = coerce_html(sibling_row['html'])

task_data = dict(mapping_result)
task_data.update({
    'html_source': sibling_html,
    'dynamic_id_enable': True,
    'dynamic_classid_enable': True,
    'more_noise_enable': True,
    'dynamic_classid_similarity_threshold': 0.85,
})

t0 = time.perf_counter()
propagated = web.layout_parser_cls({}).parse(task_data)
prop_time = time.perf_counter() - t0

prop_html = str(propagated.get('main_html_body') or '')
prop_sim = propagated.get('main_html_sim')
prop_success = propagated.get('main_html_success')

print(f"Propagation time: {prop_time:.2f}s  (no GPU used)")
print(f"Success: {prop_success}")
print(f"Similarity to template: {prop_sim:.3f}" if prop_sim else "Similarity: N/A")
print(f"Extracted HTML: {len(prop_html):,} chars")

## 8. Validation — Measure Quality vs Pure Dripper

We compare propagated output vs the LLM-extracted content using **token-level bag-of-words F1**:
- Tokenize both strings (`\w+` regex)
- Compute precision and recall over token multisets
- F1 = harmonic mean

F1=1.0 means perfect match. We target F1≥0.95 for all saved rows.

In [ ]:
from nemo_curator.stages.text.experimental.dripper.stage import _token_f1, _convert_main_html

# Convert propagated HTML to content
try:
    prop_content = _convert_main_html(bindings, prop_html, sibling_row.get('url'))
except Exception:
    prop_content = prop_html  # fallback

# Get the ground-truth LLM content from baseline
baseline_content = str(sibling_row.get('dripper_content') or '')

# Compute F1
f1 = _token_f1(str(prop_content), baseline_content)

print(f"Sibling URL: {sibling_row['url'][-80:]}")
print(f"")
print(f"Propagated content ({len(str(prop_content))} chars):")
print(str(prop_content)[:400])
print()
print(f"Baseline LLM content ({len(baseline_content)} chars):")
print(baseline_content[:400])
print()
print(f"Token F1: {f1:.4f} {'✅ PASS' if f1 >= 0.95 else '❌ FAIL (below 0.95)'})")

In [ ]:
# Measure F1 across all pages in the cluster
f1_scores = []
for _, row in demo_cluster.iterrows():
    sibling_html_i = coerce_html(row['html'])
    task_i = dict(mapping_result)
    task_i.update({'html_source': sibling_html_i,
                   'dynamic_id_enable': True, 'dynamic_classid_enable': True,
                   'more_noise_enable': True, 'dynamic_classid_similarity_threshold': 0.85})
    try:
        prop_i = web.layout_parser_cls({}).parse(task_i)
        prop_content_i = _convert_main_html(bindings, str(prop_i.get('main_html_body') or ''), row.get('url'))
        baseline_i = str(row.get('dripper_content') or '')
        f1_i = _token_f1(str(prop_content_i), baseline_i)
        f1_scores.append({'url': row['url'], 'f1': f1_i, 'error': ''})
    except Exception as e:
        f1_scores.append({'url': row['url'], 'f1': 0.0, 'error': str(e)[:80]})

f1_df = pd.DataFrame(f1_scores)
print(f"F1 distribution across {len(f1_df)} pages in cluster {demo_cluster_id}:")
print(f"  Mean F1:   {f1_df['f1'].mean():.4f}")
print(f"  Min F1:    {f1_df['f1'].min():.4f}")
print(f"  F1 ≥ 0.95: {(f1_df['f1'] >= 0.95).sum()} / {len(f1_df)} pages")
print()
print(f1_df[['url', 'f1']].to_string(index=False))

## 9. Cost Analysis — How Much GPU Time We Save

Compare layout template mode vs pure per-page Dripper:
- **Baseline**: every page needs LLM inference
- **Layout mode**: only representatives + validation + fallbacks need LLM
- **Propagated rows**: CPU only (no H100 needed)

In [ ]:
# Summarize global cluster statistics
vc = manifest[manifest['dripper_layout_id'].str.startswith('layout-', na=False)]['dripper_layout_id'].value_counts()

total_pages = len(manifest)
clustered_pages = len(manifest[manifest['dripper_layout_id'].str.startswith('layout-', na=False)])
standalone_pages = total_pages - clustered_pages
n_clusters = len(vc)

# In layout mode: ~1 representative + 2 validation rows per cluster
rep_calls = n_clusters  # one representative per cluster
val_calls = n_clusters * 2  # 2 validation LLM calls per cluster
propagated = clustered_pages - rep_calls - val_calls
total_llm_in_layout_mode = rep_calls + val_calls + standalone_pages
call_reduction = 1 - (total_llm_in_layout_mode / total_pages)

print("=" * 60)
print("COST ANALYSIS — 8192 pages from CC-MAIN-2025-26")
print("=" * 60)
print(f"Total pages:              {total_pages:>6,}")
print(f"")
print("Pure Dripper (baseline):")
print(f"  LLM calls needed:       {total_pages:>6,}  (every page)")
print(f"  Throughput:             21.9 pages/s")
print(f"  Projected H100-hours:   241,993")
print(f"")
print("Layout Template mode:")
print(f"  Clustered pages:        {clustered_pages:>6,}  ({clustered_pages/total_pages*100:.1f}%)")
print(f"  Standalone (no layout): {standalone_pages:>6,}  ({standalone_pages/total_pages*100:.1f}%)")
print(f"  Layout clusters:        {n_clusters:>6,}")
print(f"  Representative calls:   {rep_calls:>6,}")
print(f"  Validation calls:       {val_calls:>6,}")
print(f"  Propagated (CPU only):  {propagated:>6,}")
print(f"  Total LLM calls:        {total_llm_in_layout_mode:>6,}")
print(f"  Call reduction:         {call_reduction*100:.1f}%")
print(f"")
print("Latest measured run (330654):")
print(f"  Actual call reduction:  26.0%")
print(f"  Saved mean F1:          0.9871")
print(f"  Projected H100-hours:   387,447")
print(f"  (Layout is still slower due to CPU propagation bottleneck)")
print(f"")
print("With deferred propagation (in progress):")
print(f"  GPU stage removes 23,859s of CPU propagation")
print(f"  Projected H100-hours:   ~160,000  (34% below baseline!)")

In [ ]:
# Visualize the savings
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 5))

configs = ['Pure Dripper\n(baseline)', 'Layout+Validation\n(best so far)', 'Deferred Propagation\n(in progress)']
h100h = [241993, 387447, 160000]
colors = ['#d9534f', '#f0ad4e', '#5cb85c']

bars = ax.bar(configs, h100h, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
ax.axhline(241993, color='#d9534f', linestyle='--', alpha=0.5, label='Pure Dripper baseline')

for bar, val in zip(bars, h100h):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Projected H100-hours (full CC snapshot)')
ax.set_title('Dripper H100-hour Cost Reduction Progress\n(CC-MAIN-2025-26, ~2.4B pages)')
ax.set_ylim(0, 500000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 10. Full Pipeline — End-to-End on This Machine

Now let's run the complete `DripperHTMLExtractionPipelineStage` on a small subset (50 pages) using the A100 GPU on this machine. This exercises the full path:
preprocess → layout clustering → representative LLM → validation → propagation → postprocess

In [ ]:
# Start vLLM server (run in background terminal, or use subprocess)
# Model: opendatalab/MinerU-HTML-v1.1-hunyuan0.5B-compact
# On A100: tensor_parallel_size=1, ~3GB VRAM

MODEL = "opendatalab/MinerU-HTML-v1.1-hunyuan0.5B-compact"
VLLM_PORT = 8100
HF_CACHE = "/raid/vjawa/hf_cache"  # reuse existing cache

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.4",
    "--max-model-len", "8192",
    "--disable-log-requests",
    "--download-dir", HF_CACHE,
]
print("vLLM start command:")
print(" ".join(vllm_cmd))
print()
print("Run this in a terminal, then come back and run the next cell.")
print(f"Server will listen on http://localhost:{VLLM_PORT}")

In [ ]:
# Or launch it here (takes ~60s to start)
import subprocess, time as _time

vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'HF_HOME': HF_CACHE, 'TRANSFORMERS_CACHE': HF_CACHE},
)
print(f"vLLM started (pid={vllm_proc.pid}). Waiting for health check...")

import urllib.request
for attempt in range(60):
    _time.sleep(2)
    try:
        urllib.request.urlopen(f'http://localhost:{VLLM_PORT}/health', timeout=2)
        print(f"✅ vLLM ready after {attempt*2}s")
        break
    except Exception:
        if attempt % 5 == 0:
            print(f"  ... still starting ({attempt*2}s)")
else:
    print("❌ vLLM did not start in 120s — check logs")

In [ ]:
# Run the full pipeline on 50 pages
from nemo_curator.stages.text.experimental.dripper import DripperHTMLExtractionPipelineStage
from nemo_curator.models.client.llm_client import AsyncOpenAIClient, GenerationConfig
from nemo_curator.tasks import DocumentBatch

CLIENT_ENDPOINT = f"http://localhost:{VLLM_PORT}/v1"

# Take 50 pages: mix of clustered (hysplitbbs) and standalone (gen.medium)
test_pages = pd.concat([
    manifest[manifest['url_host_name'] == 'hysplitbbs.arl.noaa.gov'].head(30),
    manifest[manifest['url_host_name'] == 'gen.medium.com'].head(20),
]).reset_index(drop=True)
test_pages['html'] = test_pages['html'].apply(lambda x: x.decode('utf-8', errors='replace') if isinstance(x, bytes) else str(x))

client = AsyncOpenAIClient(
    base_url=CLIENT_ENDPOINT,
    api_key="not-needed",
    model_name=MODEL,
)

stage = DripperHTMLExtractionPipelineStage(
    client=client,
    model_name=MODEL,
    html_col='html',
    url_col='url',
    host_col='url_host_name',
    layout_id_col='dripper_layout_id',
    layout_template_mode=True,
    layout_cluster_threshold=0.95,
    layout_template_validation_rows=1,
    layout_template_validation_min_content_f1=0.90,
    layout_template_validation_signature_mode='url_low_card_query_shape_item_count_exact',
    layout_template_more_noise_enable=True,
    layout_template_min_content_length_ratio=0.25,
    layout_template_max_content_length_ratio=4.0,
    layout_template_fallback_llm=True,
    max_concurrent_requests=32,
    health_check=False,
    generation_config=GenerationConfig(max_tokens=512, temperature=0.0),
)
stage.setup()

print(f"Processing {len(test_pages)} pages...")
t0 = time.perf_counter()
batch = DocumentBatch.from_pandas(test_pages)
result = stage.process(batch)
elapsed = time.perf_counter() - t0

result_df = result.to_pandas()
print(f"Done in {elapsed:.1f}s ({len(result_df)/elapsed:.1f} pages/s)")

In [ ]:
# Summarise results
n_prop = result_df.get('dripper_layout_propagated', pd.Series(False)).sum()
n_llm = result_df.get('dripper_layout_standalone_llm', pd.Series(False)).sum() + \
        result_df.get('dripper_layout_fallback_llm', pd.Series(False)).sum()
n_rep  = result_df.get('dripper_layout_representative', pd.Series(False)).sum()
n_err  = (result_df.get('dripper_error', pd.Series('')).fillna('') != '').sum()

print("=" * 50)
print(f"RESULTS — {len(result_df)} pages")
print("=" * 50)
print(f"  Representatives (LLM):     {n_rep}")
print(f"  Propagated (CPU only):     {n_prop}  ← no GPU call!")
print(f"  Standalone/fallback (LLM): {n_llm}")
print(f"  Errors:                    {n_err}")
print(f"  Speed:                     {len(result_df)/elapsed:.1f} pages/s")
print()

# Show sample extracted content
content_col = 'dripper_content'
if content_col in result_df.columns:
    sample_results = result_df[result_df[content_col].notna() & (result_df[content_col] != '')].head(3)
    for _, r in sample_results.iterrows():
        prop_label = '(propagated)' if r.get('dripper_layout_propagated') else '(LLM)'
        print(f"URL: {r['url'][-70:]}  {prop_label}")
        print(f"Content: {str(r[content_col])[:200].strip()}")
        print()

## Summary

| Step | What it does | Cost |
|------|-------------|------|
| DOM feature extraction | Per-depth tag bag from lxml | CPU, ~5ms/page |
| Layout clustering (DBSCAN) | Groups structurally similar pages | CPU, ~50ms/cluster |
| Representative selection | Picks best-coverage page | CPU, ~20ms/cluster |
| HTML simplification | Strips to 12% of original | CPU, ~50ms/page |
| LLM extraction | Labels nodes main/other | GPU, ~2-7s/page |
| Template propagation | Applies labels to siblings | CPU, ~11s/page (bottleneck!) |
| Validation | F1 vs LLM on 2 samples | CPU + GPU, ~2s overhead/cluster |

**The deferred propagation fix** (latest, job 332432) moves the 11s/page CPU cost completely off the H100 critical path — turning a 600s GPU job into a ~250s GPU job + parallel CPU job. Projected to cut H100-hours from 387K → ~160K for the full snapshot.